# 04 · Train a tiny transformer, generate text and validate the experiment

**Transformer Models · Notebook 4 of 4**

## What you will build

Everything comes together here. You assemble your pieces into a one-layer causal character model and train it on a
generated grammar with a rule that needs context: the verb must agree with its subject even when another noun sits in
between ("the cats near the dog **like** the hen."). You select the checkpoint on validation loss and report test loss
and perplexity against two baselines, then evaluate the rule itself, slice by slice. Next, you write greedy,
temperature and top-k decoding, measure what context costs and what a key/value cache saves, and finish with the capstone checklist.

## Learning objectives

By the end of this notebook you will be able to:

1. assemble the forward pass: embeddings and positions, masked blocks, a final normalisation and vocabulary logits;
2. train with Adam and gradient clipping, keep the checkpoint with the lowest validation loss, and report test
   cross-entropy and perplexity against uniform and bigram baselines;
3. evaluate the rule you care about (agreement across an intervening noun) rather than only the average loss;
4. implement greedy, temperature and top-k decoding, and measure their effect on generated sentences;
5. measure how attention's cost grows with context, and verify that a key/value cache gives the same logits with far
   fewer multiplications.

## Dependencies

**Tier 1: NumPy and Matplotlib.** The model, training and decoding use NumPy and the tiny tensor engine (hidden recap
cell); `re`, `time` and `contextlib` come from the standard library. One plotting cell (the loss curves in section 5)
uses Matplotlib, and nothing later depends on it. No internet access is needed. Training takes 3 to 5 seconds with
CPython on a laptop and several times longer in the browser; the whole notebook takes about 10 seconds with CPython.

These are small teaching models on generated data. The results below describe this grammar only; they say nothing
about natural language.

### How to use this notebook

Each building block follows the same rhythm: a short explanation, a tiny demo, then **✍️ Your turn**, where you type a function into a stub and run the check cell under it. The check prints ✅ when your function matches the reference and ❌, with the failing input, when it does not. Until your version passes, the notebook quietly uses the reference, so *Run all* always works; once it passes, every later cell runs on **your** code. Cells marked 🧪 are experiments: change the marked setting and run them again.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours
import re
import time
from contextlib import contextmanager

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged. `Tensor` supports `+ - * / ** @`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, NumPy-style indexing, `softmax`, `log_softmax` and `backward()`; next to it are `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been checked against numerical gradients. Run the cell, and open it if you want to see again how an operation passes its gradient back.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

### One more checker: `practice_tensor`

`practice` compares plain values, but most functions in this course take and return `Tensor`s. `practice_tensor` (hidden below) runs your function and the reference on the same inputs and compares two things: the values they return, and the gradients that flow back to every input that requires one. A function that slips into raw NumPy (through `.data`, `np.exp` or `float()`, for example) can return the right numbers while cutting the gradient path, so training would silently stop updating some weights. The second comparison catches that.

In [ ]:
# ── practice_tensor: practice() for functions built from Tensor operations ──


def _tensors_in(value):
    """Every Tensor inside value, looking into tuples, lists and dicts."""
    if isinstance(value, Tensor):
        return [value]
    if isinstance(value, dict):
        value = list(value.values())
    if isinstance(value, (list, tuple)):
        return [tensor for item in value for tensor in _tensors_in(item)]
    return []


def _values_and_gradients(function, args):
    """Run function(*args). Return its output arrays and the gradients it sends back to each input Tensor."""
    inputs = [tensor for tensor in _tensors_in(args) if tensor.requires_grad]
    zero_grad(inputs)
    result = function(*args)
    outputs = list(result) if isinstance(result, tuple) else [result]
    for output in outputs:
        if not isinstance(output, Tensor):
            hint = ("; a NumPy array on the left of + or @ turns the result into an object array, so put the Tensor first"
                    if getattr(output, "dtype", None) == object else "; build the result from Tensor operations")
            raise TypeError(f"expected a Tensor but got {type(output).__name__}{hint}")
    probe = Tensor(0.0)
    for output in outputs:  # an uneven weighting, so that no gradient cancels out by symmetry
        probe = probe + (output * np.cos(np.arange(output.data.size)).reshape(output.shape)).sum()
    probe.backward()
    values = [output.data.copy() for output in outputs]
    gradients = [np.zeros_like(tensor.data) if tensor.grad is None else tensor.grad.copy() for tensor in inputs]
    zero_grad(inputs)
    return values, gradients


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """Like practice(), for functions built from Tensor operations.

    It checks the values your function returns AND the gradients that flow back
    through them, so code that slips into raw NumPy (which cuts the gradient
    path) does not pass. Returns yours once it matches, otherwise the reference.
    """
    try:
        for args in cases:
            got, got_gradients = _values_and_gradients(yours, args)
            want, want_gradients = _values_and_gradients(reference, args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got[0] if len(got) == 1 else tuple(got))}")
                print(f"   expected     {_short(want[0] if len(want) == 1 else tuple(want))}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
            if not _close(got_gradients, want_gradients, atol):
                print(f"❌ {name}: the values match, but the gradients flowing back through your function do not.")
                print("   Use Tensor operations only: .data, np.* functions and float() cut the gradient path.")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: values and gradients match the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. "
          f"From here on the notebook runs YOUR {name}.")
    return yours

## 📦 Recap: what you built earlier

The hidden cell defines your pieces from notebooks 01–03 under their usual names: the tokenizer (`encode`, `decode`),
embeddings and positions, attention with its masks and heads, `layer_norm`, `mlp`, `transformer_block`, `init_block`,
`next_token_pairs` and `masked_token_loss`.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–03, under their usual names.


def encode(text, stoi):
    """Return the list of token ids for the characters of text, in order."""
    return [stoi[character] for character in text]


def decode(ids, itos):
    """Return the string spelled by a sequence of token ids."""
    return "".join(itos[index] for index in ids)


def embedding_lookup(table, ids):
    """Rows of the embedding table for each id. table: Tensor (vocab, width); ids: ints of any shape."""
    return table[ids]


def sinusoidal_positions(length, width):
    """Sinusoidal position signals, shape (length, width); width must be even."""
    positions = np.arange(length)[:, None]
    rates = 1.0 / 10000 ** (np.arange(0, width, 2) / width)
    signals = np.zeros((length, width))
    signals[:, 0::2] = np.sin(positions * rates)
    signals[:, 1::2] = np.cos(positions * rates)
    return signals


def swap_last_two(t):
    """(..., L, d) -> (..., d, L). The engine's transpose takes the full order of axes."""
    axes = list(range(t.ndim))
    axes[-2], axes[-1] = axes[-1], axes[-2]
    return t.transpose(*axes)


def attention(q, k, v, mask=None):
    """Scaled dot-product attention. Returns (output, weights)."""
    scores = q @ swap_last_two(k) / np.sqrt(q.shape[-1])
    if mask is not None:
        scores = scores + np.where(mask, 0.0, -1e9)
    weights = scores.softmax(axis=-1)
    return weights @ v, weights


def causal_mask(length):
    """Boolean (length, length): True where key position <= query position."""
    return np.tril(np.ones((length, length), dtype=bool))


def padding_mask(token_ids, pad_id):
    """Boolean (batch, 1, length): True where the key is a real token, not padding."""
    return (np.asarray(token_ids) != pad_id)[:, None, :]


def split_heads(x, n_heads):
    """(batch, length, width) -> (batch, heads, length, width // heads)."""
    batch, length, width = x.shape
    return x.reshape(batch, length, n_heads, width // n_heads).transpose(0, 2, 1, 3)


def merge_heads(x):
    """(batch, heads, length, head_width) -> (batch, length, heads * head_width)."""
    batch, heads, length, head_width = x.shape
    return x.transpose(0, 2, 1, 3).reshape(batch, length, heads * head_width)


def init_attention(width, rng):
    """Query, key, value and output projections, each (width, width)."""
    return {name: parameter(width, width, rng=rng, scale=width ** -0.5) for name in ("Wq", "Wk", "Wv", "Wo")}


def multi_head_attention(x, params, n_heads, mask=None):
    """Self-attention with n_heads heads. x: (B, L, D). mask: boolean, broadcastable to (B, L, L).
    Returns (output (B, L, D), weights (B, heads, L, L))."""
    q = split_heads(x @ params["Wq"], n_heads)
    k = split_heads(x @ params["Wk"], n_heads)
    v = split_heads(x @ params["Wv"], n_heads)
    if mask is not None:
        mask = np.asarray(mask)[..., None, :, :]  # add a head axis: (L, L) -> (1, L, L); (B, L, L) -> (B, 1, L, L)
    out, weights = attention(q, k, v, mask)
    return merge_heads(out) @ params["Wo"], weights


def layer_norm(x, gamma, beta, eps=1e-5):
    """Normalise each position's feature vector (the last axis), then scale and shift."""
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    variance = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (variance + eps) ** 0.5 * gamma + beta


def mlp(x, params):
    """Position-wise feed-forward network: ReLU(x W1 + b1) W2 + b2."""
    return (x @ params["W1"] + params["b1"]).relu() @ params["W2"] + params["b2"]


def transformer_block(x, params, n_heads, mask=None):
    """Pre-norm block: x + attention(LN1(x)), then + MLP(LN2(.))."""
    attended, _ = multi_head_attention(layer_norm(x, params["ln1_g"], params["ln1_b"]), params, n_heads, mask)
    x = x + attended
    return x + mlp(layer_norm(x, params["ln2_g"], params["ln2_b"]), params)


def init_block(width, hidden, rng, n_blocks=1):
    """Parameters of one pre-norm block. Output projections start small so that a deep stack starts calm."""
    params = init_attention(width, rng)
    params["Wo"].data *= (2 * n_blocks) ** -0.5
    params.update({
        "ln1_g": Tensor(np.ones(width), requires_grad=True), "ln1_b": zeros_parameter(width),
        "ln2_g": Tensor(np.ones(width), requires_grad=True), "ln2_b": zeros_parameter(width),
        "W1": parameter(width, hidden, rng=rng), "b1": zeros_parameter(hidden),
        "W2": parameter(hidden, width, rng=rng, scale=(hidden * 2 * n_blocks) ** -0.5), "b2": zeros_parameter(width),
    })
    return params


def next_token_pairs(token_ids):
    """Inputs and targets for next-token prediction: shift by exactly one position."""
    token_ids = np.asarray(token_ids)
    return token_ids[:, :-1], token_ids[:, 1:]


def masked_token_loss(logits, targets, mask):
    """Mean negative log-likelihood over real target tokens. logits: (B, L, V); targets, mask: (B, L)."""
    log_probs = logits.log_softmax(axis=-1).reshape(-1, logits.shape[-1])
    targets = np.asarray(targets).reshape(-1)
    picked = log_probs[np.arange(targets.size), targets]
    weights = np.asarray(mask, dtype=float).reshape(-1)
    return -(picked * weights).sum() / weights.sum()

## 1 · The data: a grammar with a long-range rule

Every sentence follows one template, and its verb agrees in number with its subject:

- "the cat **likes** the dog." and "the cats **like** the dog.";
- with an optional phrase between them: "the cats near the dog **like** the hen.".

When the noun in the phrase (the **attractor**) differs in number from the subject, the nearest noun points to the
wrong verb form. The deciding letter, the plural "s" of the subject, can be 15 characters back, far beyond what a bigram
model (which only sees the previous character) can use. Only attention can carry it forward.

**Splitting by source.** Sentences built from the same (subject, verb, object) triple share most of their characters.
To keep near-copies out of the evaluation, the triples are split first: 70 triples for training, 15 for validation and 15
for test. Each triple gives 12 sentences (both numbers for the subject and the object, each with no phrase, an agreeing
attractor and a differing attractor), so validation and test measure new combinations of known words. That makes 840
training sentences, well within the course's data budget.

In [ ]:
NOUNS = ["cat", "dog", "bird", "frog", "hen"]
VERBS = ["like", "help", "find", "call"]


def make_sentence(subject, plural, verb, obj, object_plural, attractor=None):
    """attractor: None, or (noun, plural) for a "near the ..." phrase after the subject."""
    words = ["the", subject + "s" * plural]
    if attractor is not None:
        words += ["near", "the", attractor[0] + "s" * attractor[1]]
    words += [verb + "s" * (1 - plural), "the", obj + "s" * object_plural]
    return " ".join(words) + "."


def make_splits(seed=0):
    """Split by (subject, verb, object) triple, so that validation and test sentences are new combinations."""
    rng = np.random.default_rng(seed)
    triples = [(s, v, o) for s in NOUNS for v in VERBS for o in NOUNS]
    order = rng.permutation(len(triples))
    chosen = {"train": order[:70], "validation": order[70:85], "test": order[85:]}
    splits = {}
    for name, indices in chosen.items():
        sentences = []
        for index in indices:
            subject, verb, obj = triples[index]
            for plural in (0, 1):
                for object_plural in (0, 1):
                    same, other = rng.choice(len(NOUNS), size=2)
                    sentences.append(make_sentence(subject, plural, verb, obj, object_plural))
                    sentences.append(make_sentence(subject, plural, verb, obj, object_plural, (NOUNS[same], plural)))
                    sentences.append(make_sentence(subject, plural, verb, obj, object_plural, (NOUNS[other], 1 - plural)))
        splits[name] = [sentences[i] for i in rng.permutation(len(sentences))]
    return splits, {name: {triples[i] for i in indices} for name, indices in chosen.items()}


splits, split_triples = make_splits(seed=0)
for name, sentences in splits.items():
    print(f"{name:10s} {len(split_triples[name]):3d} triples {len(sentences):4d} sentences, e.g. {sentences[0]!r}")

## 2 · Vocabulary, rows and the context limit

The tokenizer is your character tokenizer from notebook 01, with its vocabulary built from the **training** sentences.
A newline, `"\n"`, marks both the start and the end of every sentence, so each row is `"\n" + sentence + "\n"`, padded
with `<pad>` (id 0) to the longest row. The model learns to emit `"\n"` when a sentence is complete, which tells
generation where to stop. The **context limit** is the longest input the model sees in training; generation crops
its input to that many tokens.

In [ ]:
itos = ["<pad>"] + sorted(set("".join(splits["train"])) | {"\n"})
stoi = {token: index for index, token in enumerate(itos)}
PAD, V = 0, len(itos)


def to_rows(sentences):
    """Encode "\n" + sentence + "\n" for each sentence and pad with PAD to the longest: an int array (B, L)."""
    encoded = [encode("\n" + sentence + "\n", stoi) for sentence in sentences]
    rows = np.full((len(encoded), max(map(len, encoded))), PAD)
    for r, ids in enumerate(encoded):
        rows[r, :len(ids)] = ids
    return rows


data = {name: to_rows(sentences) for name, sentences in splits.items()}
CONTEXT = max(rows.shape[1] for rows in data.values()) - 1
print(f"{V} tokens: {''.join(itos[1:])!r} (plus <pad>)")
print({name: rows.shape for name, rows in data.items()}, "| context limit:", CONTEXT, "tokens")

## 3 · ✍️ Assemble the model

The whole model is a few lines on top of your pieces:

$$x = E[\mathrm{ids}] + PE, \qquad x \leftarrow \mathrm{block}_\ell(x, \text{mask}) \text{ for each layer } \ell, \qquad \mathrm{logits} = \mathrm{LN}_f(x)\, W_{\text{out}} + b_{\text{out}} .$$

**Shapes.** ids $(B, L)$ → embeddings plus positions $(B, L, D)$ → the same shape after every block → logits
$(B, L, V)$, one score per vocabulary entry at every position. The mask is `causal_mask(L) & padding_mask(ids, 0)`,
of shape $(B, L, L)$. A final layer norm before the output projection is standard in pre-norm models, because the
residual stream itself is never normalised.

`init_model` (given) creates the parameters, and `parameters_of` lists every `Tensor` in that nested structure, for
the optimiser.

In [ ]:
def init_model(vocab_size, width, n_layers, hidden, rng):
    """All parameters in one dict; "blocks" holds one init_block dict per layer."""
    return {
        "tok": parameter(vocab_size, width, rng=rng, scale=1.0),
        "blocks": [init_block(width, hidden, rng, n_blocks=n_layers) for _ in range(n_layers)],
        "lnf_g": Tensor(np.ones(width), requires_grad=True), "lnf_b": zeros_parameter(width),
        "W_out": parameter(width, vocab_size, rng=rng, scale=width ** -0.5), "b_out": zeros_parameter(vocab_size),
    }


def parameters_of(params):
    """Every Tensor in a nested dict or list of parameters, in a fixed order."""
    if isinstance(params, Tensor):
        return [params]
    values = params.values() if isinstance(params, dict) else params
    return [tensor for value in values for tensor in parameters_of(value)]


tiny = init_model(7, 8, 1, 16, np.random.default_rng(0))
tiny["b_out"].data += np.random.default_rng(1).normal(size=7)  # a non-zero bias, so that forgetting it shows
print("tiny model:", sum(p.data.size for p in parameters_of(tiny)), "numbers in", len(parameters_of(tiny)), "tensors")

### ✍️ Your turn: `forward`

Write `forward(params, ids, n_heads)` for a model from `init_model` and an integer array `ids` of shape `(B, L)` that
uses id 0 for padding. Return the logits, a `Tensor` of shape `(B, L, vocab)`:

1. `x = embedding_lookup(params["tok"], ids) + sinusoidal_positions(L, D)`, where `D = params["tok"].shape[1]`;
2. `mask = causal_mask(L) & padding_mask(ids, 0)`;
3. for every `block` in `params["blocks"]`: `x = transformer_block(x, block, n_heads, mask)`;
4. `x = layer_norm(x, params["lnf_g"], params["lnf_b"])`;
5. return `x @ params["W_out"] + params["b_out"]`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def forward_reference(params, ids, n_heads):
    """Logits (B, L, vocab) for token ids (B, L), padding id 0."""
    length = ids.shape[1]
    x = embedding_lookup(params["tok"], ids) + sinusoidal_positions(length, params["tok"].shape[1])
    mask = causal_mask(length) & padding_mask(ids, 0)
    for block in params["blocks"]:
        x = transformer_block(x, block, n_heads, mask)
    x = layer_norm(x, params["lnf_g"], params["lnf_b"])
    return x @ params["W_out"] + params["b_out"]

In [ ]:
def forward(params, ids, n_heads):
    """Logits (B, L, vocab) for token ids (B, L), padding id 0."""
    # ✍️ Your code here (about seven lines)
    raise NotImplementedError

In [ ]:
ids_tiny = np.array([[1, 3, 4, 2, 0], [1, 5, 6, 6, 2]])
forward = practice_tensor("forward", forward, forward_reference, cases=[
    (tiny, ids_tiny, 2),
    (tiny, np.array([[1]]), 4),
])

**Expected behaviour.** The check compares the logits and the gradients of every parameter, block weights included.
Before training anything, the next cell repeats notebook 02's proof on the whole model: change every token from
position 12 onwards, and the logits before position 12 must not move.

In [ ]:
N_HEADS, WIDTH, N_LAYERS, HIDDEN = 4, 32, 1, 64  # 🧪 N_LAYERS = 2 roughly doubles the training time


def causal_leak(model, rows, change_from, seed=0):
    """Largest change in the logits before change_from when every token from change_from onwards is replaced."""
    changed = rows.copy()
    changed[:, change_from:] = np.random.default_rng(seed).integers(1, V, size=changed[:, change_from:].shape)
    before = forward(model, rows, N_HEADS).data[:, :change_from]
    after = forward(model, changed, N_HEADS).data[:, :change_from]
    return np.abs(before - after).max()


model = init_model(V, WIDTH, N_LAYERS, HIDDEN, np.random.default_rng(0))
params = parameters_of(model)
probe_rows = data["validation"][:8, :-1]
print(f"{sum(p.data.size for p in params):,} parameters")
print("logits before position 12 move by", causal_leak(model, probe_rows, 12), "when every later token changes")
assert causal_leak(model, probe_rows, 12) < 1e-12

## 4 · Baselines and a one-batch sanity check

Two baselines frame the result:

- a **uniform** guess over the $V - 1$ real tokens has loss $\log(V - 1)$;
- a **bigram** model predicts the next character from the current one using counts from the training rows (with a
  little add-$\alpha$ smoothing). It learns spelling and word order, but it cannot see an "s" fifteen characters back.

Before a long run, check that the model can **overfit one small batch**. If the loss on four sentences does not fall
close to zero within a few dozen steps, something in the model, the loss or the optimiser is broken, and training on
the full data would only hide it.

The optimiser is **Adam** (Kingma and Ba, 2015): each weight takes a step scaled by running averages of its gradient and
squared gradient. Frameworks often use AdamW, which adds decoupled weight decay; this small run does without.
`clip_gradients` scales all gradients down together whenever their overall norm exceeds a limit, a common guard
against a single bad batch.

In [ ]:
def evaluate(model, rows):
    """Mean next-token loss over the real targets of rows."""
    inputs, targets = next_token_pairs(rows)
    return float(masked_token_loss(forward(model, inputs, N_HEADS), targets, targets != PAD).data)


def fit_bigram(rows, alpha=0.1):
    """P(next | current) from counts of (current, next) pairs in the rows, with add-alpha smoothing."""
    counts = np.full((V, V), alpha)
    inputs, targets = next_token_pairs(rows)
    real = targets != PAD
    np.add.at(counts, (inputs[real], targets[real]), 1.0)
    return counts / counts.sum(axis=1, keepdims=True)


def bigram_loss(probs, rows):
    inputs, targets = next_token_pairs(rows)
    real = targets != PAD
    return float(-np.log(probs[inputs[real], targets[real]]).mean())


class Adam:
    """Adam: per-weight steps from running averages of the gradient and of its square."""

    def __init__(self, params, lr=5e-3, beta1=0.9, beta2=0.99, eps=1e-8):
        self.params, self.lr, self.beta1, self.beta2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in params]
        self.v = [np.zeros_like(p.data) for p in params]
        self.t = 0

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[:] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[:] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat, v_hat = m / (1 - self.beta1 ** self.t), v / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


def clip_gradients(params, max_norm=1.0):
    """Scale all gradients down together when their overall norm exceeds max_norm; return the norm before clipping."""
    norm = float(np.sqrt(sum((p.grad ** 2).sum() for p in params if p.grad is not None)))
    if norm > max_norm:
        for p in params:
            if p.grad is not None:
                p.grad *= max_norm / norm
    return norm


uniform_loss = np.log(V - 1)
bigram = fit_bigram(data["train"])
print(f"uniform guess:  loss {uniform_loss:.3f}   perplexity {V - 1}")
print(f"bigram counts:  validation loss {bigram_loss(bigram, data['validation']):.3f}   perplexity {np.exp(bigram_loss(bigram, data['validation'])):.2f}")
print(f"fresh model:    validation loss {evaluate(model, data['validation']):.3f}")

In [ ]:
sanity_model = init_model(V, WIDTH, N_LAYERS, HIDDEN, np.random.default_rng(1))
sanity_params = parameters_of(sanity_model)
sanity_optimiser = Adam(sanity_params, lr=1e-2)
inputs, targets = next_token_pairs(data["train"][:4])
sanity_losses = []
for step in range(60):
    loss = masked_token_loss(forward(sanity_model, inputs, N_HEADS), targets, targets != PAD)
    zero_grad(sanity_params)
    loss.backward()
    sanity_optimiser.step()
    sanity_losses.append(float(loss.data))
print("loss on one batch of 4 sentences, every 10 steps:", np.round(sanity_losses[::10], 3), "->", round(sanity_losses[-1], 4))
assert sanity_losses[-1] < 0.1 * sanity_losses[0], "the model should be able to memorise four sentences"

## 5 · Train, keeping the best validation checkpoint

Each step samples 32 training sentences, drops columns that are padding in every row, computes your masked loss,
backpropagates, clips and updates. Every 50 steps the model is scored on the validation sentences, and the weights with
the lowest validation loss so far are copied aside. The test sentences are not touched during training. A copy of the
weights at step 100 is kept too, for the failure analysis in section 6.

In [ ]:
steps = 500             # 🧪 try 150 or 1000 (the floors in section 10 assume the defaults)
learning_rate = 5e-3
batch_size = 32

model = init_model(V, WIDTH, N_LAYERS, HIDDEN, np.random.default_rng(0))
params = parameters_of(model)
optimiser = Adam(params, lr=learning_rate)
batch_rng = np.random.default_rng(1)
history, early_weights = [], None
best = {"loss": np.inf, "step": 0, "weights": None}
initial_validation_loss = evaluate(model, data["validation"])

start = time.perf_counter()
for step in range(1, steps + 1):
    rows = data["train"][batch_rng.integers(0, len(data["train"]), batch_size)]
    rows = rows[:, :(rows != PAD).sum(axis=1).max()]
    inputs, targets = next_token_pairs(rows)
    real = targets != PAD
    assert real.any(), "a batch without real targets would divide by zero"
    loss = masked_token_loss(forward(model, inputs, N_HEADS), targets, real)
    zero_grad(params)
    loss.backward()
    clip_gradients(params, 1.0)
    optimiser.step()
    if step % 50 == 0:
        validation_loss = evaluate(model, data["validation"])
        history.append((step, float(loss.data), validation_loss))
        if validation_loss < best["loss"]:
            best = {"loss": validation_loss, "step": step, "weights": [p.data.copy() for p in params]}
    if step == 100:
        early_weights = [p.data.copy() for p in params]
train_seconds = time.perf_counter() - start


def load_weights(params, weights):
    for p, saved in zip(params, weights):
        p.data[...] = saved


print(f"{steps} steps in {train_seconds:.1f} s; best validation loss {best['loss']:.3f} at step {best['step']}")
print(" step   training loss   validation loss")
for step, training_loss, validation_loss in history:
    print(f"{step:5d}   {training_loss:13.3f}   {validation_loss:15.3f}")

In [ ]:
steps_logged = [h[0] for h in history]
plt.figure(figsize=(6.5, 3.4))
plt.plot(steps_logged, [h[1] for h in history], label="training batch")
plt.plot(steps_logged, [h[2] for h in history], label="validation")
plt.axhline(bigram_loss(bigram, data["validation"]), color="grey", linestyle="--", label="bigram (validation)")
plt.axvline(best["step"], color="black", linestyle=":", label=f"kept: step {best['step']}")
plt.xlabel("step"); plt.ylabel("mean loss per real token (nats)"); plt.ylim(0, 1.5); plt.legend()
plt.title("loss falls far below the bigram baseline")
plt.tight_layout()
plt.show()

## 6 · Evaluate: perplexity, baselines and the rule itself

Load the kept checkpoint and score the **test** sentences once. **Perplexity** is $\exp$ of the mean loss in nats: the
effective number of equally likely choices per character. Uniform guessing has perplexity $V - 1 = 20$.

Average loss is dominated by easy positions: spelling "the" and the other fixed letters. The rule we care about is
decided at **one** position per sentence, the character after the verb stem: "s" for a singular subject, a space for
a plural one. `agreement_accuracy` checks, at that position, whether the correct ending scores higher than the wrong
one (an exact tie counts as half right, so a model with no preference scores the chance level of 0.5), separately for
each kind of sentence. Two reference points help: the **nearest-noun heuristic**,
which copies the number of the closest noun, and the bigram model.

In [ ]:
def decision_point(sentence):
    """Where the verb's ending is decided: (input position, correct next character, kind of sentence)."""
    words = sentence[:-1].split(" ")
    verb_index = 5 if words[2] == "near" else 2
    prefix = " ".join(words[:verb_index]) + " " + words[verb_index][:4]  # every verb stem has four letters
    if verb_index == 2:
        kind = "no attractor"
    elif words[1].endswith("s") == words[4].endswith("s"):
        kind = "attractor agrees"
    else:
        kind = "attractor differs"
    return len(prefix), sentence[len(prefix)], kind


def agreement_accuracy(score_rows, sentences):
    """Per kind of sentence: how often the correct verb ending outscores the wrong one at the decision point."""
    rows = to_rows(sentences)
    scores = score_rows(rows[:, :-1])
    hits = {}
    for r, sentence in enumerate(sentences):
        position, correct, kind = decision_point(sentence)
        wrong = " " if correct == "s" else "s"
        right_score, wrong_score = scores[r, position, stoi[correct]], scores[r, position, stoi[wrong]]
        hits.setdefault(kind, []).append(1.0 if right_score > wrong_score else 0.5 if right_score == wrong_score else 0.0)
    return {kind: float(np.mean(h)) for kind, h in sorted(hits.items())}


def nearest_noun_accuracy(sentences):
    hits = {}
    for sentence in sentences:
        words = sentence[:-1].split(" ")
        _, correct, kind = decision_point(sentence)
        nearest = words[4] if words[2] == "near" else words[1]
        hits.setdefault(kind, []).append((" " if nearest.endswith("s") else "s") == correct)
    return {kind: float(np.mean(h)) for kind, h in sorted(hits.items())}


def model_scores(rows):
    return forward(model, rows, N_HEADS).data


load_weights(params, early_weights)
early_agreement = agreement_accuracy(model_scores, splits["test"])
load_weights(params, best["weights"])
final_agreement = agreement_accuracy(model_scores, splits["test"])
bigram_agreement = agreement_accuracy(lambda rows: np.log(bigram[rows]), splits["test"])
heuristic_agreement = nearest_noun_accuracy(splits["test"])

test_loss = evaluate(model, data["test"])
bigram_test_loss = bigram_loss(bigram, data["test"])
print(f"test loss {test_loss:.3f} nats per character, perplexity {np.exp(test_loss):.2f}")
print(f"bigram    {bigram_test_loss:.3f}, perplexity {np.exp(bigram_test_loss):.2f};  uniform {uniform_loss:.3f}, perplexity {V - 1}")
print()
print("agreement on the 180 test sentences (60 of each kind); chance = 0.50")
print(f"{'kind':18s} {'nearest noun':>12s} {'bigram':>7s} {'step 100':>9s} {'kept model':>11s}")
for kind in final_agreement:
    print(f"{kind:18s} {heuristic_agreement[kind]:12.2f} {bigram_agreement[kind]:7.2f} "
          f"{early_agreement[kind]:9.2f} {final_agreement[kind]:11.2f}")

**What to notice.** The nearest-noun heuristic is perfect on two kinds and always wrong on "attractor differs"; a
model that learned that shortcut would still look good on average. The bigram model cannot see the subject at all. At
step 100 the model's loss had already fallen a long way, yet "attractor differs" was its weakest kind: a low average
loss can hide the one failure that matters. The kept model scores at or near 1.00 on every kind. Only attention can
have carried the subject's "s" across the phrase, because no other part of the model moves information between
positions.

## 7 · ✍️ Decoding: greedy, temperature and top-k

Generation repeats one step: run the model on the text so far, take the **last** position's logits $z \in
\mathbb{R}^V$, choose a token, append it, and stop at the end marker or a length limit. How you choose is the decoding
strategy, and it changes the output without changing a single weight:

- **greedy**: take $\arg\max_i z_i$, which is deterministic;
- **temperature** $T$: sample from $p_i = e^{z_i/T} / \sum_j e^{z_j/T}$. $T < 1$ sharpens the distribution towards
  greedy and $T > 1$ flattens it towards uniform;
- **top-k**: keep only the $k$ largest logits (the rest become $-\infty$, so their probability is 0) before sampling.

`sample_from` (given) draws one id per row by inverse-CDF sampling: it draws $u \sim U(0, 1)$ and returns the first
index whose cumulative probability exceeds $u$. It takes a generator object, but your functions below stay
deterministic, so `practice` can compare them.

In [ ]:
def sample_from(probs, rng):
    """One id per row of probs (B, V): the first index whose cumulative probability exceeds a uniform draw."""
    u = rng.random((probs.shape[0], 1))
    return np.minimum((np.cumsum(probs, axis=-1) < u).sum(axis=-1), probs.shape[-1] - 1)


logits_demo = np.array([[2.0, 1.0, 0.0, -1.0]])
for T in (0.5, 1.0, 2.0):
    scaled = np.exp(logits_demo / T)
    print(f"T = {T}: probabilities {np.round(scaled / scaled.sum(), 3)}")

### ✍️ Your turn: `greedy_next`

Write `greedy_next(logits)` for a NumPy array of shape `(B, V)`. Return an integer array of shape `(B,)` with the index
of each row's largest logit (if several tie, the first).

*Hint:* `np.argmax` with an `axis`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def greedy_next_reference(logits):
    """The id of the largest logit in each row. logits: (B, V) -> ints (B,)."""
    return np.argmax(logits, axis=-1)

In [ ]:
def greedy_next(logits):
    """The id of the largest logit in each row. logits: (B, V) -> ints (B,)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
logits_check = np.array([[0.1, 2.0, -1.0, 0.5], [1.5, -0.5, 3.0, 0.0]])
greedy_next = practice("greedy_next", greedy_next, greedy_next_reference, cases=[
    (logits_check,),
    (np.array([[3.0, 3.0, 1.0]]),),  # a tie: the first index wins
])

### ✍️ Your turn: `temperature_probs`

Write `temperature_probs(logits, temperature)`. Divide the logits by `temperature` and apply a softmax along the last
axis. Subtract each row's maximum before `np.exp`, so that large logits cannot overflow. Entries equal to `-np.inf`
(from top-k) must come out as probability 0.

*Hint:* `scaled - scaled.max(axis=-1, keepdims=True)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def temperature_probs_reference(logits, temperature):
    """Softmax of logits / temperature along the last axis."""
    scaled = np.asarray(logits, dtype=float) / temperature
    scaled = scaled - scaled.max(axis=-1, keepdims=True)
    exps = np.exp(scaled)
    return exps / exps.sum(axis=-1, keepdims=True)

In [ ]:
def temperature_probs(logits, temperature):
    """Softmax of logits / temperature along the last axis."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
temperature_probs = practice("temperature_probs", temperature_probs, temperature_probs_reference, cases=[
    (logits_check, 1.0),
    (logits_check, 0.25),
    (logits_check, 4.0),
    (np.array([[1.0, -np.inf, 0.0]]), 1.0),
])

### ✍️ Your turn: `top_k_filter`

Write `top_k_filter(logits, k)`. Keep each row's `k` largest logits and replace the others by `-np.inf`. (If several
logits tie with the k-th largest, keep them all.) Return a new array; do not change `logits`.

*Hints:* `np.sort(logits, axis=-1)[..., -k]` is each row's k-th largest value; add an axis with `[..., None]` and use
`np.where`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def top_k_filter_reference(logits, k):
    """Keep each row's k largest logits (and any ties with the k-th); set the rest to -inf."""
    logits = np.asarray(logits, dtype=float)
    kth_largest = np.sort(logits, axis=-1)[..., -k][..., None]
    return np.where(logits >= kth_largest, logits, -np.inf)

In [ ]:
def top_k_filter(logits, k):
    """Keep each row's k largest logits (and any ties with the k-th); set the rest to -inf."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
top_k_filter = practice("top_k_filter", top_k_filter, top_k_filter_reference, cases=[
    (logits_check, 2),
    (logits_check, 1),
    (logits_check, 4),  # keeps everything
])

**Expected behaviour.** `top_k_filter(logits, 1)` followed by any temperature gives the same choice as greedy.
`generate` below (given) runs the loop. Its `choose` argument is any function from `(logits, rng)` to the next ids, so
greedy and sampling share one loop.

In [ ]:
END = stoi["\n"]


def generate(model, prompts, choose, max_new_tokens=45, seed=0):
    """Extend equal-length prompts one token at a time. choose(logits (B, V), rng) returns the next ids (B,).
    Each text is cut at the first end marker after its prompt."""
    rng = np.random.default_rng(seed)
    rows = np.array([encode(prompt, stoi) for prompt in prompts])
    start = rows.shape[1]
    for _ in range(max_new_tokens):
        logits = forward(model, rows[:, -CONTEXT:], N_HEADS).data[:, -1]  # crop to the context limit
        rows = np.concatenate([rows, np.asarray(choose(logits, rng))[:, None]], axis=1)
        if (rows[:, start:] == END).any(axis=1).all():
            break
    texts = []
    for prompt, row in zip(prompts, rows):
        texts.append((prompt + decode(row[start:], itos)).lstrip("\n").split("\n")[0])
    return texts


def greedy(logits, rng):
    return greedy_next(logits)


def sampler(temperature, k=None):
    def choose(logits, rng):
        if k is not None:
            logits = top_k_filter(logits, k)
        return sample_from(temperature_probs(logits, temperature), rng)
    return choose


NOUN = "(cat|dog|bird|frog|hen)"
GRAMMAR = re.compile(rf"the {NOUN}(?P<plural>s?)( near the {NOUN}s?)? (like|help|find|call)(?P<verb_s>s?) the {NOUN}s?\.")


def judge(text):
    """'agrees' for a grammatical sentence whose verb agrees with its subject, 'disagrees', or 'invalid'."""
    match = GRAMMAR.fullmatch(text)
    if match is None:
        return "invalid"
    return "agrees" if (match["plural"] == "s") != (match["verb_s"] == "s") else "disagrees"


print("greedy:", generate(model, ["\n"], greedy)[0])
for text in generate(model, ["\n"] * 4, sampler(1.0), seed=3):
    print(f"T = 1.0: {text!r:52} {judge(text)}")
print("prompted, greedy:", generate(model, ["\nthe cats near the hen "], greedy)[0])

### Agreement in generated text

Teacher-forced accuracy (section 6) scores one position with the correct text in front of it. Generation has to produce
the ending itself. The cell prompts the kept model with every subject followed by an attractor of the *other* number,
"the cats near the dog ", lets greedy decoding finish the sentence, and judges the result.

In [ ]:
prompts = [f"\nthe {s}{'s' * p} near the {a}{'s' * (1 - p)} " for s in NOUNS for p in (0, 1) for a in NOUNS]
by_length = {}
for prompt in prompts:
    by_length.setdefault(len(prompt), []).append(prompt)  # generate() needs prompts of equal length
completions = [text for group in by_length.values() for text in generate(model, group, greedy, max_new_tokens=25)]
verdicts = [judge(text) for text in completions]
generated_agreement = verdicts.count("agrees") / len(verdicts)
print(f"{len(prompts)} prompts with a differing attractor: {verdicts.count('agrees')} agree, "
      f"{verdicts.count('disagrees')} disagree, {verdicts.count('invalid')} invalid")
print("for example:", completions[:3])

## 8 · The cost of context, and the key/value cache

A full forward pass over $L$ tokens forms an $L \times L$ score matrix in every head of every layer, so doubling the
context **quadruples** the score entries. Optimised kernels such as FlashAttention compute the same dense attention
without storing the whole matrix at once, but the arithmetic still grows with $L^2$.

Generation adds another cost. The loop in `generate` re-runs the model on the whole prefix at every step, recomputing
keys and values that have not changed. A **key/value (KV) cache** stores each layer's keys and values; each new step
computes the query, key and value of the new position only and attends over the cache. Everything else in a block (layer
norms, the MLP, residuals) works on one position at a time, so the result is the same. The cache holds
$2 \times \text{layers} \times L \times D$ numbers per sequence, and that memory grows with batch size, context and
depth.

`counting_multiplications` counts the scalar multiplications inside every `Tensor` matrix product run within it; the
remaining operations (exponentials, sums) are cheap by comparison and are not counted.

In [ ]:
print(" context   score entries per head and layer   float32 memory for one matrix")
for length in (128, 1024, 8192, 65536):
    print(f"{length:8d}   {length ** 2:32,d}   {length ** 2 * 4 / 2 ** 20:14.1f} MiB")


@contextmanager
def counting_multiplications():
    """Count the scalar multiplications inside every Tensor matrix product run in this block."""
    counter = {"multiplications": 0}
    plain_matmul = Tensor.__matmul__

    def counted_matmul(self, other):
        other = Tensor.lift(other)
        batch = int(np.prod(np.broadcast_shapes(self.shape[:-2], other.shape[:-2])))
        counter["multiplications"] += batch * self.shape[-2] * self.shape[-1] * other.shape[-1]
        return plain_matmul(self, other)

    Tensor.__matmul__ = counted_matmul
    try:
        yield counter
    finally:
        Tensor.__matmul__ = plain_matmul


def forward_step(model, token_ids, position, cache, n_heads):
    """Logits (B, V) for one new token per row at `position`, reusing the keys and values in `cache`.
    cache: one {"k": array or None, "v": array or None} dict per block; this call appends to it."""
    width = model["tok"].shape[1]
    x = embedding_lookup(model["tok"], token_ids[:, None]) + sinusoidal_positions(position + 1, width)[position]
    for block, stored in zip(model["blocks"], cache):
        h = layer_norm(x, block["ln1_g"], block["ln1_b"])
        q = split_heads(h @ block["Wq"], n_heads)  # (B, heads, 1, head width): the new query only
        k_new = split_heads(h @ block["Wk"], n_heads).data
        v_new = split_heads(h @ block["Wv"], n_heads).data
        stored["k"] = k_new if stored["k"] is None else np.concatenate([stored["k"], k_new], axis=2)
        stored["v"] = v_new if stored["v"] is None else np.concatenate([stored["v"], v_new], axis=2)
        out, _ = attention(q, Tensor(stored["k"]), Tensor(stored["v"]))  # no mask: the cache holds only the past
        x = x + merge_heads(out) @ block["Wo"]
        x = x + mlp(layer_norm(x, block["ln2_g"], block["ln2_b"]), block)
    x = layer_norm(x, model["lnf_g"], model["lnf_b"])
    return (x @ model["W_out"] + model["b_out"]).data[:, 0]


def run_both_ways(tokens):
    """Logits for every prefix of tokens: recomputed from scratch, then with the cache. Returns both, with counts."""
    with counting_multiplications() as full:
        full_logits = np.array([forward(model, tokens[None, :t + 1], N_HEADS).data[0, -1] for t in range(len(tokens))])
    cache = [{"k": None, "v": None} for _ in model["blocks"]]
    with counting_multiplications() as cached:
        cached_logits = np.array([forward_step(model, tokens[t:t + 1], t, cache, N_HEADS)[0] for t in range(len(tokens))])
    return full_logits, cached_logits, full["multiplications"], cached["multiplications"]


longest = to_rows([max(splits["test"], key=len)])[0, :-1]
full_logits, cached_logits, full_count, cached_count = run_both_ways(longest)
cache_gap = np.abs(full_logits - cached_logits).max()
print(f"\n{len(longest)} generation steps: largest logit difference {cache_gap:.1e}")
print(f"multiplications: {full_count:,} recomputing, {cached_count:,} with the cache ({full_count / cached_count:.1f} times fewer)")
assert cache_gap < 1e-9, "the cache must reproduce the full computation"

## 9 · 🧪 Experiments

### Decoding settings change the text, not the model

Sixty sentences per setting, all from the same kept weights and the same seed. The table reports how many are
grammatical with correct agreement, grammatical with the wrong verb form, or not sentences of the grammar at all, and
how many distinct sentences appear.

In [ ]:
settings = [(0.5, None), (1.0, None), (1.5, None), (2.5, None), (1.5, 3), (2.5, 3)]  # 🧪 add (1.0, 1) or (5.0, None)
print(" temperature  top-k   agrees  disagrees  invalid  distinct")
sampling_results = {}
for temperature, k in settings:
    texts = generate(model, ["\n"] * 60, sampler(temperature, k), seed=1)
    verdicts = [judge(text) for text in texts]
    sampling_results[(temperature, k)] = verdicts.count("agrees") / len(texts)
    print(f"{temperature:12.1f}  {str(k):>5}   {verdicts.count('agrees'):6d}  {verdicts.count('disagrees'):9d}  "
          f"{verdicts.count('invalid'):7d}  {len(set(texts)):8d}")
print("a high-temperature sample:", generate(model, ["\n"], sampler(2.5), seed=2)[0])

**What to notice.** Low temperature gives almost only valid, agreeing sentences, but with less variety; on a larger
grammar it would repeat itself. High temperature produces broken words, because it lends real probability to letters the
model rates as unlikely. Top-k cuts off that tail and recovers much of the validity at the same temperature. None of these
settings changed a weight, so report the decoding settings whenever you show generated text.

### How the saving grows with length

Recomputing costs grow roughly with the square of the number of generated tokens; the cache grows roughly linearly.

In [ ]:
lengths = (10, 20, len(longest))  # 🧪 the model was trained on at most CONTEXT tokens; stay within it
print(" tokens   recomputing   with cache   ratio")
for n in lengths:
    _, _, full_n, cached_n = run_both_ways(longest[:n])
    print(f"{n:7d}   {full_n:11,d}   {cached_n:10,d}   {full_n / cached_n:5.1f}")

## 10 · Put it together: the capstone checklist

A small transformer experiment is only finished when another person can check it. Each item below is produced by the
next two cells:

1. **Tokenizer and data:** the vocabulary, the end marker, the padding id, the context limit and a split manifest
   showing that no triple or sentence appears in two splits.
2. **Architecture with shapes:** a trace of one batch through the model.
3. **Correctness checks:** future tokens do not influence earlier logits in the trained model, targets are the inputs
   shifted once, and padded targets are not scored.
4. **Selection and evaluation:** the validation history and the kept step, then test loss and perplexity beside the
   uniform and bigram baselines, measured once.
5. **The rule and a failure analysis:** agreement by kind of sentence, for the kept model and for the step-100
   checkpoint.
6. **Generation, reported separately:** decoding settings and samples, kept apart from the quantitative evaluation.

In [ ]:
# Item 2, architecture: one batch through the model, with shapes.
trace_ids = data["test"][:2, :-1]
L = trace_ids.shape[1]
block = model["blocks"][0]
x = embedding_lookup(model["tok"], trace_ids) + sinusoidal_positions(L, WIDTH)
mask = causal_mask(L) & padding_mask(trace_ids, PAD)
q = split_heads(layer_norm(x, block["ln1_g"], block["ln1_b"]) @ block["Wq"], N_HEADS)
print(f"ids {trace_ids.shape} -> embeddings + positions {x.shape}; mask {mask.shape}")
print(f"queries per head {q.shape}; attention weights {(2, N_HEADS, L, L)}; MLP hidden {(2, L, HIDDEN)}")
print(f"after {N_LAYERS} block(s) {x.shape} -> final layer norm -> logits {forward(model, trace_ids, N_HEADS).shape}")

# Item 3, correctness checks on the trained model.
trained_leak = causal_leak(model, data["test"][:16, :-1], 15)
inputs, targets = next_token_pairs(data["test"])
logits = forward(model, inputs, N_HEADS)
scrambled = np.where(targets == PAD, np.random.default_rng(5).integers(1, V, size=targets.shape), targets)
padding_scored = abs(float(masked_token_loss(logits, scrambled, targets != PAD).data)
                     - float(masked_token_loss(logits, targets, targets != PAD).data))

# Items 1, 4, 5 and 6: the report.
report = {
    "task": "generated agreement grammar with attractors (a teaching task, not natural language)",
    "vocabulary": "".join(itos[1:]).replace("\n", "\\n") + " (+ <pad> = 0; \\n marks start and end)",
    "context_limit": CONTEXT,
    "splits": {name: f"{len(split_triples[name])} triples, {len(splits[name])} sentences" for name in splits},
    "model": f"{N_LAYERS} pre-norm block(s), width {WIDTH}, {N_HEADS} heads, MLP {HIDDEN}, sinusoidal positions, "
             f"{sum(p.data.size for p in params):,} parameters",
    "training": f"Adam lr {learning_rate}, batch {batch_size}, {steps} steps, clip 1.0, seed 0, {train_seconds:.1f} s",
    "kept_step": best["step"], "validation_loss": round(best["loss"], 4),
    "test_loss": round(test_loss, 4), "test_perplexity": round(float(np.exp(test_loss)), 3),
    "bigram_test_perplexity": round(float(np.exp(bigram_test_loss)), 3), "uniform_perplexity": V - 1,
    "agreement_test_kept": final_agreement, "agreement_test_step_100": early_agreement,
    "generated_agreement_with_differing_attractor": round(generated_agreement, 3),
    "decoding": "greedy, and sampling at T in {0.5, 1.0, 1.5, 2.5} with and without top-3, seed 1",
    "kv_cache": f"logits equal to {cache_gap:.0e}; {full_count / cached_count:.1f} times fewer multiplications over {len(longest)} steps",
}
for key, value in report.items():
    print(f"{key:46s} {value}")

In [ ]:
triples = list(split_triples.values())
assert all(not (a & b) for i, a in enumerate(triples) for b in triples[i + 1:]), "no triple may appear in two splits"
assert not (set(splits["train"]) & (set(splits["validation"]) | set(splits["test"]))), "no sentence may be shared"
assert trained_leak < 1e-12, "future tokens must not change earlier logits"
assert np.array_equal(inputs[:, 1:], targets[:, :-1]), "targets must be the inputs shifted once"
assert padding_scored < 1e-12, "padded targets must not be scored"
assert best["loss"] < 0.5 * initial_validation_loss, "training should cut the validation loss by more than half"
assert test_loss < 0.6 * bigram_test_loss, f"test loss {test_loss:.3f} should be well below the bigram's {bigram_test_loss:.3f}"
assert final_agreement["attractor differs"] >= 0.8, f"agreement across a differing attractor is {final_agreement['attractor differs']:.2f}"
assert generated_agreement >= 0.8, f"greedy completions agree in only {generated_agreement:.2f} of cases"
assert sampling_results[(0.5, None)] >= 0.75, "most samples at temperature 0.5 should be valid and agree"
assert cache_gap < 1e-9 and cached_count * 5 < full_count, "the cache must match and save most multiplications"
print("✅ Capstone checklist complete: every check passed on this run.")
print(f"   Test perplexity {np.exp(test_loss):.2f} (bigram {np.exp(bigram_test_loss):.2f}, uniform {V - 1}); agreement across a")
print(f"   differing attractor {final_agreement['attractor differs']:.2f} (step 100: {early_agreement['attractor differs']:.2f}; nearest-noun heuristic 0.00).")

## Exercises

### ✍️ Your turn: `top_p_filter`

**Exercise 1: nucleus (top-p) sampling.** Instead of a fixed number of tokens, keep the smallest set of most likely
tokens whose total probability reaches `p` (Holtzman et al., 2020). Write `top_p_filter(logits, p)`: compute the
probabilities with `temperature_probs(logits, 1.0)`, visit tokens from most to least likely, keep a token while the
probability mass *before* it is still below `p` (so the token that crosses `p` is kept), and set the rest to `-np.inf`.

*Hints:* `np.argsort(-probs, axis=-1)` gives the order; `np.cumsum` minus the probabilities themselves gives the mass
before each token; `np.put_along_axis` writes a result back in the original order.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def top_p_filter_reference(logits, p):
    """Keep the smallest set of most likely tokens whose probability reaches p; set the rest to -inf."""
    logits = np.asarray(logits, dtype=float)
    probs = temperature_probs(logits, 1.0)
    order = np.argsort(-probs, axis=-1, kind="stable")
    sorted_probs = np.take_along_axis(probs, order, axis=-1)
    mass_before = np.cumsum(sorted_probs, axis=-1) - sorted_probs
    keep_sorted = mass_before < p
    keep = np.zeros_like(keep_sorted)
    np.put_along_axis(keep, order, keep_sorted, axis=-1)
    return np.where(keep, logits, -np.inf)

In [ ]:
def top_p_filter(logits, p):
    """Keep the smallest set of most likely tokens whose probability reaches p; set the rest to -inf."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
logits_p = np.log(np.array([[0.5, 0.3, 0.15, 0.05], [0.05, 0.1, 0.6, 0.25]]))
top_p_filter = practice("top_p_filter", top_p_filter, top_p_filter_reference, cases=[
    (logits_p, 0.7),  # keeps 0.5 and 0.3 in row 0, and 0.6 and 0.25 in row 1
    (logits_p, 0.9),
    (logits_p, 1.0),
])

**Exercise 2: a longer dependency.** Write sentences with *two* attractors, such as "the cats near the dog near the hen
like the bird.", which the model never saw in training. Before you measure anything, predict the agreement accuracy. Then
adapt `decision_point` and measure it. Some of these sentences are longer than the context limit of training; how would
you separate a failure of the rule from a failure of length?

**Exercise 3: depth.** Set `N_LAYERS = 2` in section 3 and run sections 3–6 again. Compare the step-100 agreement, the
kept checkpoint's results and the training time. Does the second layer pay for itself on this task?

**Exercise 4: your own rule (capstone extension).** Add a construction to the grammar, for example coordinated subjects
("the cat and the dog like ..."). Decide the evaluation slices and the baseline that targets the rule *before* you
train, keep the split by source, and rerun the checklist.

## Common errors and debugging

- **Generation that ignores the context limit.** Positions beyond those seen in training were never learned. Crop the
  input to `CONTEXT` tokens, as `generate` does, and record the limit with the checkpoint.
- **Temperature 0.** Dividing by zero gives `inf` or `nan`. Use greedy decoding as the limit $T \to 0$.
- **`k` larger than the vocabulary in top-k.** `np.sort(...)[..., -k]` raises `IndexError`. Clamp `k` to `V`.
- **Choosing the checkpoint or the temperature on test data.** Then the test score is no longer an unbiased measure.
  Select on validation data, and score test data once.
- **Comparing perplexities across tokenizers or corpora.** A character model's perplexity is per character; a subword
  model's is per subword token. The numbers are not comparable.
- **A KV-cache position off by one.** The new token's position is the number of tokens already in the cache. The
  equality check in section 8 catches this.
- **A loss that falls while samples look wrong.** Check the shift, the masks and the end marker, then evaluate the
  positions that test the rule you care about, as section 6 does.

## Summary

- The model is embeddings plus positions, masked pre-norm blocks, a final layer norm and an output projection; with
  9,845 parameters it learns this grammar in about 500 steps.
- Validation loss chose the checkpoint, and test data was scored once: the test perplexity is far below the bigram and
  uniform baselines.
- Evaluating the rule by kind of sentence exposes what the average loss hides, such as the step-100 model's weakness
  across a differing attractor.
- Greedy, temperature and top-k decoding trade validity against variety without changing any weight.
- Attention's score matrix grows with the square of the context; a KV cache gives the same logits (to about
  $10^{-14}$) with many fewer multiplications during generation.

## Next

The Large Language Model Training course scales these mechanics up: pretraining corpora and their provenance,
instruction tuning with response-only losses, and parameter-efficient adaptation. What you measured here (masks, token
losses, baselines and evaluation by slice) carries over unchanged.